### Simple Self-Attention Implementation Code (PyTorch)

In [ ]:

import torch
import torch.nn.functional as F   # torch.nn.functional (usually imported as F) is a module that contains functions for neural networks.

# example input(static embedding): 3 words, embedding size = 4// i love cats  
X = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],     # Word 1: "I"
    [0.0, 2.0, 0.0, 2.0],     # Word 2: "love"

    [1.0, 1.0, 1.0, 1.0]      # Word 3: "cats"
])
# in big dataset, at first nn.embedding creates fot every word vector of d dimensions.embedding_layer = nn.Embedding(vocab_size, embed_dim)

d = X.shape[1]      # embedding dimension We extract the embedding dimension (4) for scaling later
                    #shape[1] gives the second dimension (columns)


# initialize weight matrices
Wq = torch.rand(d, d)        # We create three random matrices for Query, Key, Value
                             #In real transformers: These are learned during training
Wk = torch.rand(d, d)
Wv = torch.rand(d, d)

# compute Q, K, V
Q = X @ Wq                # We multiply our input X with each weight matrix
                          #The @ symbol is matrix multiplication (dot product)
                          #Each word gets its own Query, Key, and Value vector
K = X @ Wk
V = X @ Wv

# attention scores
scores = Q @ K.T          # Result is a [3, 3] matrix showing relationships between all words.Dot product measures similarity. If Q[i] and K[j] point in similar directions, they have a high score.

                          #scores[0, 1] = How much should Word 1 ("I") pay attention to Word 2 ("love")?
                          #scores[2, 0] = How much should Word 3 ("cats") pay attention to Word 1 ("I")?


# scale scores
scores = scores / (d ** 0.5)       # To prevent extremely small gradients during training
                                   # Without scaling, dot products can get very large, pushing softmax into regions with tiny gradients

# softmax
attention_weights = F.softmax(scores, dim=1)    # [3,3] Softmax converts scores into probabilities that sum to 1
                                                # dim=1 means we apply softmax across columns (for each row)

# final output
output = attention_weights @ V         # Each word's final representation now contains: Its own information (from its Value)
                                       # PLUS context from other words (weighted by attention)  
          
print("Attention Weights:\n", attention_weights)
print("Output:\n", output)

Attention Weights:
 tensor([[0.1774, 0.2473, 0.5753],
        [0.1678, 0.1363, 0.6959],
        [0.1129, 0.1418, 0.7453]])
Output:
 tensor([[0.9961, 1.2972, 0.9601, 1.2066],
        [1.0880, 1.3233, 1.0598, 1.2122],
        [1.0952, 1.3574, 1.0638, 1.2488]])
